In [1]:
import pandas as pd
import numpy as np
import os
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

# Path setup sesuai struktur folder kamu
BASE_DIR = ".."
DATA_PATH = os.path.join(BASE_DIR, "data")
MODEL_PATH = os.path.join(BASE_DIR, "models")
SUB_PATH = os.path.join(BASE_DIR, "submissions")

print("Setup Ready!")

Setup Ready!


In [2]:
def load_data(file_name):
    path = os.path.join(DATA_PATH, file_name)
    # Optimasi tipe data untuk hemat RAM
    dtypes = {
        'GPS_Accuracy_M': 'float32',
        'Distance_KM': 'float32',
        'Est_Price_IDR': 'float32',
        'Surge_Multiplier': 'float32',
        'Accel_X': 'float32',
        'Accel_Y': 'float32',
        'Accel_Z': 'float32',
        'Gyro_Z': 'float32'
    }
    return pd.read_csv(path, dtype=dtypes)

df_train = load_data("train.csv")
df_test = load_data("test.csv")
print(f"Data Loaded! Train: {df_train.shape}, Test: {df_test.shape}")

Data Loaded! Train: (8000000, 25), Test: (4000000, 24)


In [3]:
def build_features(df):
    # 1. Time Features
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['hour'] = df['Timestamp'].dt.hour
    df['day_of_week'] = df['Timestamp'].dt.dayofweek
    
    # 2. Sensor Magnitude (Vector Sum untuk Safety_Violation)
    df['accel_mag'] = np.sqrt(df['Accel_X']**2 + df['Accel_Y']**2 + df['Accel_Z']**2)
    
    # 3. Price per KM (Untuk Fraud & Service_Complaint)
    # Tambah epsilon 0.001 supaya gak division by zero
    df['price_per_km'] = df['Est_Price_IDR'] / (df['Distance_KM'] + 0.001)
    
    # 4. Clean Battery (Handle string dan % di Test)
    if df['Battery_Level'].dtype == 'object':
        df['Battery_Level'] = df['Battery_Level'].astype(str).str.replace('%', '').replace('nan', np.nan)
        df['Battery_Level'] = pd.to_numeric(df['Battery_Level'], errors='coerce')
    
    # 5. Missing Info (Kadang data kosong = Fraud)
    df['null_count'] = df.isnull().sum(axis=1)
    
    # Drop yang gak perlu
    cols_to_drop = ['Timestamp']
    if 'Trip_ID' in df.columns:
        cols_to_drop.append('Trip_ID')
    
    return df.drop(columns=cols_to_drop)

# Eksekusi Feature Engineering
y = df_train['Trip_Label']
X = build_features(df_train.drop(columns=['Trip_Label']))
X_test_final = build_features(df_test)

print("Feature Engineering Done!")

Feature Engineering Done!


In [4]:
# Ambil 25% data secara stratified
X_samp, _, y_samp, _ = train_test_split(
    X, y, train_size=0.25, stratify=y, random_state=42
)

# Split untuk Validasi
X_train, X_val, y_train, y_val = train_test_split(
    X_samp, y_samp, test_size=0.15, stratify=y_samp, random_state=42
)

# Identifikasi kolom kategori untuk CatBoost
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
X_train[cat_features] = X_train[cat_features].fillna("MISSING").astype(str)
X_val[cat_features] = X_val[cat_features].fillna("MISSING").astype(str)
X_test_final[cat_features] = X_test_final[cat_features].fillna("MISSING").astype(str)

print(f"Final Train Size: {len(X_train)} rows")

Final Train Size: 1700000 rows


In [6]:
# Sesuai instruksi: gunakan Seed untuk reproduksibilitas
SEED = 42

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=8,
    task_type="CPU",
    thread_count=12,
    loss_function='MultiClass',
    # Menggunakan MultiClassCustom atau Accuracy yang lebih stabil di MultiClass
    eval_metric='Accuracy', 
    auto_class_weights='Balanced', # Tetap pakai ini untuk hajar imbalanced class
    early_stopping_rounds=50,
    random_seed=SEED,
    verbose=100
)

# Bungkus data ke Pool agar lebih efisien di RAM
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

model.fit(
    train_pool,
    eval_set=val_pool,
    use_best_model=True
)

0:	learn: 0.5826370	test: 0.5817115	best: 0.5817115 (0)	total: 22.4s	remaining: 6h 12m 8s
100:	learn: 0.5891526	test: 0.5852528	best: 0.5855096 (83)	total: 17m 39s	remaining: 2h 37m 12s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5855096056
bestIteration = 83

Shrink model to first 84 iterations.


In [7]:
# Prediksi pada data validasi
# .flatten() memastikan bentuknya array 1D
y_val_pred = model.predict(val_pool).flatten()

# Hitung Macro F1 sesuai metrik kompetisi
macro_f1 = f1_score(y_val, y_val_pred, average="macro")

print("="*50)
print(f"FINAL MACRO F1 SCORE: {macro_f1:.4f}")
print("="*50)
print("\nDETAILED CLASSIFICATION REPORT:")
# zero_division=0 supaya tidak error kalau ada kelas yang tidak terprediksi
print(classification_report(y_val, y_val_pred, zero_division=0))

FINAL MACRO F1 SCORE: 0.5447

DETAILED CLASSIFICATION REPORT:
                   precision    recall  f1-score   support

 Fraud_Indication       0.74      1.00      0.85     15012
 Navigation_Issue       0.13      0.17      0.15     30067
     Perfect_Trip       0.73      0.51      0.60    164910
 Safety_Violation       0.92      0.97      0.95     60060
Service_Complaint       0.13      0.27      0.18     29951

         accuracy                           0.57    300000
        macro avg       0.53      0.59      0.54    300000
     weighted avg       0.65      0.57      0.59    300000



In [8]:
test_preds = model.predict(X_test_final)
submission = pd.DataFrame({
    "Trip_ID": df_test["Trip_ID"],
    "Trip_Label": test_preds.flatten()
})

sub_file = os.path.join(SUB_PATH, "submission_final_kafi_1.csv")
submission.to_csv(sub_file, index=False)
print(f"Submission saved to: {sub_file}")

Submission saved to: ../submissions/submission_final_kafi_1.csv
